# Distance heatmap

Targets × datasets, restricted to targets observed in all 4 datasets and to `type == "targeting"`. Three panels:

- **raw**: `distance_mean` as-is (color scale dominated by Huangfu's 100-1000× larger values — see calibration caveat in [README](README.md))
- **rank**: per-column rank, scaled to [0, 1]
- **quantile**: each column mapped onto the average sorted profile (proper quantile normalization)

Rows are clustered (hierarchical, on rank-normalized values).

**Input:** `results/distance_heatmap/per_target_wide.tsv` (from `scripts/build_wide_per_target_table.py`)
**Output:** `results/distance_heatmap/distance_heatmap.pdf`

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

PROJECT_ROOT = Path("/cellar/users/aklie/projects/tf_perturb_seq")
sys.path.append(str(PROJECT_ROOT / "config"))
from loader import load_colors

EDIST = PROJECT_ROOT / "docs/jamborees/2026_UTSW/working_groups/wg1_data_qc/edist"
RESULTS = EDIST / "results" / "distance_heatmap"
RESULTS.mkdir(parents=True, exist_ok=True)

dataset_colors = load_colors("production_TF-Perturb-seq", "dataset_colors")

SHORT = {
    "Hon_WTC11-cardiomyocyte-differentiation_TF-Perturb-seq": "HonCM",
    "Huangfu_HUES8-definitive-endoderm-differentiation_TF-Perturb-seq": "HuangfuDE",
    "Huangfu_HUES8-embryonic-stemcell-differentiation_TF-Perturb-seq": "HuangfuESC",
    "Gersbach_WTC11-hepatocyte-differentiation_TF-Perturb-seq": "GersbachHep",
    "Engreitz_WTC11-endothelial-cells_TF-Perturb-seq": "EngreitzEndo",
}
SHORT_TO_FULL = {v: k for k, v in SHORT.items()}
DATASET_ORDER = ["HonCM", "HuangfuDE", "HuangfuESC", "GersbachHep"]
COLORS = {s: dataset_colors[SHORT_TO_FULL[s]] for s in DATASET_ORDER}

print("RESULTS:", RESULTS)


In [ ]:
from scipy.cluster.hierarchy import linkage, leaves_list
from scipy.spatial.distance import pdist

wide = pd.read_csv(RESULTS / "per_target_wide.tsv", sep="\t")
ds_cols = [f"{s}_distance_mean" for s in DATASET_ORDER]
type_cols = [f"{s}_type" for s in DATASET_ORDER]

mask = wide[type_cols].eq("targeting").all(axis=1) & wide[ds_cols].notna().all(axis=1)
mat = wide.loc[mask, ["target_id", "gene_symbol"] + ds_cols].copy()
mat.columns = ["target_id", "gene_symbol"] + DATASET_ORDER
mat = mat.set_index("gene_symbol")[DATASET_ORDER]
print(f"heatmap matrix: {mat.shape[0]} targets × {mat.shape[1]} datasets")

def rank_normalize(df: pd.DataFrame) -> pd.DataFrame:
    return df.rank(axis=0, method="average") / len(df)

def quantile_normalize(df: pd.DataFrame) -> pd.DataFrame:
    sorted_each = pd.DataFrame(np.sort(df.values, axis=0), columns=df.columns)
    ref = sorted_each.mean(axis=1).values
    rank = df.rank(axis=0, method="min").astype(int) - 1
    out = pd.DataFrame(ref[rank.values], index=df.index, columns=df.columns)
    return out

variants = {"raw": mat, "rank": rank_normalize(mat), "quantile": quantile_normalize(mat)}

row_link = linkage(pdist(variants["rank"].values, metric="euclidean"), method="average")
row_order = leaves_list(row_link)
for k in variants:
    variants[k] = variants[k].iloc[row_order]

fig, axes = plt.subplots(1, 3, figsize=(13, 9))
for ax, (label, m) in zip(axes, variants.items()):
    vmax = m.values.max()
    im = ax.imshow(m.values, aspect="auto", cmap="viridis",
                   vmin=0, vmax=vmax, interpolation="nearest")
    ax.set_xticks(range(m.shape[1]))
    ax.set_xticklabels(m.columns, rotation=45, ha="right")
    ax.set_yticks([])
    ax.set_title(label)
    fig.colorbar(im, ax=ax, fraction=0.04, pad=0.02)
fig.suptitle(f"Distance heatmap — {mat.shape[0]} shared targeting targets")
plt.tight_layout()
fig.savefig(RESULTS / "distance_heatmap.pdf")
plt.show()